In [1]:
import matplotlib.pyplot as plot
import pandas
import seaborn

seaborn.set_theme(style='ticks', context='paper', palette='plasma')

LEADS_PATH, PEDIDOS_PATH = '../data/churn_aplicacao_leads_final.csv', '../data/churn_comportamental_pedidos_final.csv'

In [2]:
leads = pandas.read_csv(LEADS_PATH)
pedidos = pandas.read_csv(PEDIDOS_PATH)

In [3]:
leads.info()

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 19 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   id_cliente                       2000 non-null   int64  
 1   uf                               2000 non-null   str    
 2   municipio                        2000 non-null   str    
 3   segmento_cliente                 2000 non-null   str    
 4   natureza_juridica                2000 non-null   str    
 5   fonte_cliente                    2000 non-null   str    
 6   cnae_codigo                      2000 non-null   str    
 7   capital_social                   2000 non-null   str    
 8   idade_cnpj                       2000 non-null   str    
 9   serasa_contagem_negativacoes     2000 non-null   float64
 10  serasa_contagem_protestos        2000 non-null   float64
 11  serasa_credores                  190 non-null    str    
 12  serasa_socio_tem_negativacao   

In [ ]:
pedidos.info()

<class 'pandas.DataFrame'>
RangeIndex: 4441 entries, 0 to 4440
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   id_pedido     4441 non-null   int64
 1   id_cliente    4441 non-null   int64
 2   valor         4438 non-null   str  
 3   produtos      4440 non-null   str  
 4   data_entrega  4441 non-null   str  
dtypes: int64(2), str(3)
memory usage: 173.6 KB


Cruzando a tabela de leads com a de pedidos:

In [5]:
joined = leads.merge(right=pedidos, on='id_cliente', how='left')

Análise estrutural dos dados:

In [6]:
joined.info()

<class 'pandas.DataFrame'>
RangeIndex: 5441 entries, 0 to 5440
Data columns (total 23 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   id_cliente                       5441 non-null   int64  
 1   uf                               5441 non-null   str    
 2   municipio                        5441 non-null   str    
 3   segmento_cliente                 5441 non-null   str    
 4   natureza_juridica                5441 non-null   str    
 5   fonte_cliente                    5441 non-null   str    
 6   cnae_codigo                      5441 non-null   str    
 7   capital_social                   5441 non-null   str    
 8   idade_cnpj                       5441 non-null   str    
 9   serasa_contagem_negativacoes     5441 non-null   float64
 10  serasa_contagem_protestos        5441 non-null   float64
 11  serasa_credores                  447 non-null    str    
 12  serasa_socio_tem_negativacao   

Porcentagem de faltantes por coluna:

In [ ]:
print('faltantes por atributo em LEADS')
for label in leads.columns:
    column: pandas.Series = leads[label]
    missing_pct = 100 * column.isna().sum() / (column.isna().sum() + column.notna().sum())
    
    if missing_pct > 0:
        print(f'{label:32s} | {missing_pct:.2f}%')

faltantes por atributo em LEADS
serasa_credores                  | 90.50%
ifood_contagem_avaliacoes        | 94.30%
ifood_faixa_preco                | 87.85%
google_maps_avaliacao            | 97.80%
google_maps_contagem_avaliacoes  | 91.25%
google_maps_tem_website          | 91.25%


In [ ]:
print('faltantes por atributo em PEDIDOS')
for label in pedidos.columns:
    column: pandas.Series = pedidos[label]
    missing_pct = 100 * column.isna().sum() / (column.isna().sum() + column.notna().sum())
    
    if missing_pct > 0:
        print(f'{label:12s} | {missing_pct:.2f}%')

faltantes por atributo em PEDIDOS
valor        | 0.07%
produtos     | 0.02%


Contando valores únicos para cada atributo:

In [83]:
for label in joined.columns:
    column: pandas.Series = joined[label]

    # Contando os valores únicos:
    unique_values = column.unique()
    print(f'{label:32s} | {unique_values.size!s:5s}', end='')

    # Imprimindo os valores possíveis para
    # os atributos somente se são poucos:
    if unique_values.size <= 20:
        values_str = ', '.join(str(v) for v in unique_values)
        print(f'({values_str})', end='')

    print()

id_cliente                       | 2000 
uf                               | 1    (CE)
municipio                        | 31   
segmento_cliente                 | 21   
natureza_juridica                | 4    (206-2 - Sociedade Empresária Limitada, 213-5 - Empresário (Individual), 230-5 - Empresa Individual de Responsabilidade Limitada (de Natureza Empresária), 224-0 - Sociedade Simples Limitada)
fonte_cliente                    | 5    (Fonte 5, Fonte 1, Fonte 3, Fonte 2, Fonte 4)
cnae_codigo                      | 107  
capital_social                   | 19   ((70000, 100000], (3000, 5000], (8000, 10000], (200000, 1000000], (15000, 20000], (50000, 70000], (30000, 50000], (5000, 8000], (100000, 200000], (20000, 30000], (1000000, 10000000], (10000, 15000], (0, 1], (1500, 2000], (200, 1000], (1, 200], (1000, 1500], (2000, 3000], (10000000, 100000000000])
idade_cnpj                       | 20   ((8000, 25000], (5000, 8000], (1300, 1600], (2000, 2400], (50, 100], (2400, 3000], (4000, 5000],

- Só tem um estado (CE)
- `google_maps_avaliacao` não tem valor no intervalo `(4.0, 4.5]`
- `capital_social`, `idade_cnpj`, `ifood_contagem_avaliacoes`, `valor` (pedido) e `produtos` (pedido) são dados em 

In [68]:
#Hierarquia do CNAE no estilo do dataset:
# AA -> divisão
# AA.B -> grupo
# AA.BC-C -> classe
# AA.BC-C-DD -> subclasse

# Contando divisões únicas
divisoes = {cnae[:2] for cnae in leads.cnae_codigo}

print(f'quantidade de divisões CNAE: {len(divisoes)} | {sorted(divisoes)}')
print(f'exemplos de alguns códigos: {sorted(leads.cnae_codigo)[:5]}')

quantidade de divisões CNAE: 27 | ['12', '14', '20', '23', '24', '25', '29', '30', '33', '36', '37', '38', '39', '52', '53', '64', '70', '74', '76', '77', '78', '81', '83', '85', '93', '96', '97']
exemplos de alguns códigos: ['12.10-3-97', '12.74-7-12', '12.74-7-32', '12.74-7-79', '14.17-3-90']


- Segundo detalhamento da IBGE/CONCLA dos [códigos CNAE 2.3](https://cnae.ibge.gov.br/images/concla/documentacao/CNAE_Subclasses_2_3_Estrutura_Detalhada.xlsx), não existe o código `12.10-3/97` por exemplo
- Não podemos contar que esses valores tenham significado explícito 
- O Notion dizia que podemos contar com a hierarquia dos códigos, mas o que isso significa?

In [75]:
print(f'quantidade de valores únicos para credores serasa: {leads.serasa_credores.nunique()}')
print(f'lista de credores: {sorted(leads.serasa_credores.dropna().unique())}')

quantidade de valores únicos para credores serasa: 49
lista de credores: ['Alimentos e Bebidas', 'Alimentos e Bebidas, Alimentos e Bebidas, Alimentos e Bebidas', 'Alimentos e Bebidas, Bancos e Financeiro', 'Alimentos e Bebidas, Bancos e Financeiro, Alimentos e Bebidas', 'Alimentos e Bebidas, Bancos e Financeiro, Bancos e Financeiro, Financeiro, Alimentos e Bebidas, Alimentos e Bebidas, Indústria', 'Alimentos e Bebidas, Saúde', 'Alimentos e Bebidas, Tecnologia e Pagamentos', 'Bancos e Financeiro', 'Bancos e Financeiro, Alimentos e Bebidas, Bancos e Financeiro', 'Bancos e Financeiro, Alimentos e Bebidas, Distribuição', 'Bancos e Financeiro, Bancos e Financeiro', 'Bancos e Financeiro, Bancos e Financeiro, Bancos e Financeiro', 'Bancos e Financeiro, Bancos e Financeiro, Telecomunicações, Contabilidade', 'Bancos e Financeiro, Comunicação', 'Bancos e Financeiro, Financeiro', 'Bancos e Financeiro, Tecnologia e Pagamentos', 'Bancos e Financeiro, Tecnologia e Pagamentos, Bancos e Financeiro', '

- Não existem 49 valores diferentes de fato, são uma lista de valores separados por vírgulas
- Existem listas com valores repetidos, pq?
- É preciso olhar valor por valor para decidir como separar os valores únicos de cada lista de fato:

In [82]:
credores: set[str] = set()

for credor in leads.serasa_credores.dropna().unique():
    credores.update(credor.split(', '))

print(f'{len(credores)} categorias únicas de credores:')
sorted(credores)

21 categorias únicas de credores:


['Alimentos e Bebidas',
 'Bancos e Financeiro',
 'Comunicação',
 'Comércio',
 'Contabilidade',
 'Distribuição',
 'Diversos',
 'Energia',
 'Financeiro',
 'Imobiliário',
 'Indústria',
 'Logística',
 'Saúde',
 'Seguros e Financeiro',
 'Serviços',
 'Serviços Ambientais',
 'Tecnologia',
 'Tecnologia e Pagamentos',
 'Telecomunicações',
 'Varejo',
 'Viagens e Turismo']

só verificando se esses são categóricos ou não msm

In [ ]:
leads.serasa_contagem_negativacoes.unique()

array([  0.,  14.,   6.,   2.,   1.,   3.,   4.,  10.,  15.,  16.,  19.,
         7.,   5.,  11.,  22.,  52.,   8.,   9., 223.,  12.,  25.,  29.,
        30.,  33.,  21., 162.,  99.])

In [86]:
leads.segmento_cliente.unique()

<StringArray>
[ 'Segmento 5', 'Segmento 19', 'Segmento 20', 'Segmento 13', 'Segmento 18',
 'Segmento 14',  'Segmento 3', 'Segmento 10', 'Segmento 15',  'Segmento 4',
 'Segmento 16',  'Segmento 8',  'Segmento 6',  'Segmento 9', 'Segmento 17',
  'Segmento 2', 'Segmento 21',  'Segmento 7',  'Segmento 1', 'Segmento 11',
 'Segmento 12']
Length: 21, dtype: str

In [ ]:
datas = pedidos.data_entrega
# pegando o período das datas de entregas
meses: set[str] = {date[:7] for date in datas.dropna().unique()}

print(f'meses: {sorted(meses)}')
print(f'quantidade de pedidos sem data de entrega: {datas.isna().sum()}')

meses: ['2024-09', '2024-10', '2024-11', '2024-12']
quantidade de pedidos sem data de entrega: 0
quantidade de entradas na tabela de leads sem pedidos: 1000


- Quando formos treinar um modelo, devemos ordenar por data e deixarmos os dados dos pedidos dos ultimos dias para teste

In [98]:
print(f'quantidade de entradas na tabela de leads sem pedidos: {joined.data_entrega.isna().sum()}')

quantidade de entradas na tabela de leads sem pedidos: 1000
